In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
# os.environ['GEMINI_API_KEY']=os.getenv("GEMINI_API_KEY")
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")
# print("GEMINI_API_KEY:", os.environ['GEMINI_API_KEY'])

from langchain.chat_models import init_chat_model
model = init_chat_model(model="llama-3.1-8b-instant", model_provider="groq",
                            api_key=os.getenv("GROQ_API_KEY"))

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."


model_with_tools = model.bind_tools([get_weather])  

response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


In [3]:
# Bind (potentially multiple) tools to the model
model_with_tools = model.bind_tools([get_weather])

# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

However, the function 'get_weather' doesn't return a real time answer. Instead it returns an answer in the following format: 

'Current weather: Sunny. High temperature: 22°C. Low temperature: 12°C.'

If you want to get more accurate weather information in real time, I suggest using the function 'get_weather' in conjunction with a web scraping tool, or by searching for real time information on a reliable weather website.


# Tool calling

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code.

In [14]:
from langchain.tools import tool
from langchain_core.messages import ToolMessage


@tool
def get_weather(location: str) -> str:
    """Get the current weather for a specified location."""
    return f"The weather in {location} is sunny with a temperature of 25°C."

model_with_tools = model.bind_tools([get_weather])
messages = [{"role": "user", "content": "What's the weather in Boston?"}]

# Call model once - it will generate tool calls
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Execute each tool call once and add results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    # Add tool result as a proper ToolMessage
    messages.append(ToolMessage(content=tool_result, tool_call_id=tool_call["id"]))

# Call model once more to get the final response with tool results
final_response = model_with_tools.invoke(messages)
print(final_response.content)

The weather in Boston is sunny with a temperature of 25°C.


In [9]:
final_response.text

''

In [ ]:

messages = [{"role": "user", "content": "What's the weather in Boston?"}]


@tool
def fetch_weather_data(location: str) -> str:
    """Fetch current weather data for a given location."""
    return f"The current weather in {location} is sunny with a temperature of 25°C."

# IMPORTANT: bind_tools returns a new runnable; you must assign it
# Try to call the model with tools; if the remote function-call fails (400),
# fall back to running the local tool directly so the notebook continues
# and later cells that inspect response.tool_calls won't error out.
from types import SimpleNamespace

model_with_tools = model.bind_tools([fetch_weather_data], tool_choice='any')

try:
    ai_msg  = model_with_tools.invoke("What's the weather in New York City?")
    messages.append(ai_msg)
except Exception as e:
    # If the model attempted to call a remote function but the platform
    # rejected it (BadRequestError/tool_use_failed), fall back to the local tool.
    print("Tool calling failed; falling back to local tool. Error:", e)
    result = fetch_weather_data("New York City")
    # Provide a minimal object that other cells expect (content and tool_calls).
    ai_msg  = SimpleNamespace(content=result, tool_calls=[], additional_kwargs={}, response_metadata={})

# Tool Calling in LangChain: Complete Guide

## What are Tools?

Tools are functions that an AI model can call to perform tasks beyond generating text. They bridge the gap between the model's knowledge and external systems like databases, APIs, calculators, or web services.

## How Tool Calling Works

### 1. **Define Tools**
Tools are Python functions decorated with `@tool` from `langchain.tools`. The docstring becomes the tool's description that the model reads:
```python
@tool
def get_weather(location: str) -> str:
    """Get current weather for a location."""
    return f"Weather in {location}: Sunny, 25°C"
```

### 2. **Bind Tools to Model**
Use `.bind_tools()` to give the model access to tools. **IMPORTANT:** This returns a new object that must be assigned:
```python
model_with_tools = model.bind_tools([get_weather, get_time], tool_choice='any')
```

### 3. **Model Processes Query**
When you invoke the model with a query, it decides whether to call a tool:
```python
response = model_with_tools.invoke("What's the weather in London?")
```

### 4. **Check Tool Calls**
Access the tool calls made by the model:
```python
response.tool_calls  # Returns list of ToolCall objects with name, args, id
```

### 5. **Execute Tools**
For each tool call, invoke the tool and collect results:
```python
for tool in response.tool_calls:
    result = get_weather.invoke(tool)  # Executes the tool
    messages.append(result)  # Add result to conversation
```

### 6. **Get Final Response**
Send tool results back to the model for a final, informed response:
```python
final_response = model.invoke(messages)
```

## Tool Design Best Practices

| Practice | Good ✅ | Bad ❌ |
|----------|---------|--------|
| **Specificity** | `multiply_numbers(a, b)` | `calculate(expression)` |
| **Docstring** | "Multiply two numbers. Use ONLY when..." | "Do math" |
| **Parameters** | `number_a: int, number_b: int` | `expression: str` |
| **Purpose** | One clear job | Generic catch-all with `eval()` |

## Common Issues & Solutions

| Issue | Cause | Solution |
|-------|-------|----------|
| Empty `tool_calls` | Result not assigned from `bind_tools()` | Use `model = model.bind_tools(...)` |
| Wrong tool called | Generic docstrings confuse the model | Use specific docstrings with "Use ONLY when..." |
| Tool not found | Whitespace in tool name | Use single-line docstrings |
| API errors (400) | Groq API limitations with tool calling | Use manual tool execution with keyword matching |

## Manual Tool Execution (Workaround)

When the LLM provider doesn't support native tool calling, implement manual execution:

```python
def execute_tool(query: str):
    if 'weather' in query.lower():
        location = extract_location(query)
        return get_weather(location)
    elif 'multiply' in query.lower():
        numbers = extract_numbers(query)
        return multiply_numbers(numbers[0], numbers[1])
```

## Key Takeaways

1. **Tools extend model capabilities** - Provide access to external data and functions
2. **Model chooses tools** - Based on query and tool descriptions
3. **Tool selection matters** - Clear, specific tools with good docstrings lead to correct choices
4. **Conversation loop** - Model → Tool Call → Tool Execution → Model → Final Response
5. **Error handling** - Always handle cases where tools fail or API limitations arise
6. **Assignment is critical** - `bind_tools()` returns new object; must be assigned to variable

# Tool Chaining 

When a model returns tool calls, you need to execute the tools and pass the results back to the model. This creates a conversation loop where the model can use tool results to generate its final response. LangChain includes agent abstractions that handle this orchestration for you.

In [12]:
messages 

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '1fjc2dft2', 'function': {'arguments': '{"location":"New York City"}', 'name': 'fetch_weather_data'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 243, 'total_tokens': 258, 'completion_time': 0.014308544, 'completion_tokens_details': None, 'prompt_time': 0.017911616, 'prompt_tokens_details': None, 'queue_time': 0.094522564, 'total_time': 0.03222016}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019be119-0106-7121-890a-372b22c7830e-0', tool_calls=[{'name': 'fetch_weather_data', 'args': {'location': 'New York City'}, 'id': '1fjc2dft2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 243, 'output_tokens': 15, 'total_tokens': 258})]

In [13]:


for tool in ai_msg.tool_calls:
    print(tool)
    tool_result = fetch_weather_data.invoke(tool)
    messages.append(tool_result)
    print("Tool result:", tool_result)

{'name': 'fetch_weather_data', 'args': {'location': 'New York City'}, 'id': '1fjc2dft2', 'type': 'tool_call'}
Tool result: content='The current weather in New York City is sunny with a temperature of 25°C.' name='fetch_weather_data' tool_call_id='1fjc2dft2'


In [14]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '1fjc2dft2', 'function': {'arguments': '{"location":"New York City"}', 'name': 'fetch_weather_data'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 243, 'total_tokens': 258, 'completion_time': 0.014308544, 'completion_tokens_details': None, 'prompt_time': 0.017911616, 'prompt_tokens_details': None, 'queue_time': 0.094522564, 'total_time': 0.03222016}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019be119-0106-7121-890a-372b22c7830e-0', tool_calls=[{'name': 'fetch_weather_data', 'args': {'location': 'New York City'}, 'id': '1fjc2dft2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 243, 'output_tokens': 15, 'total_tokens': 258}),
 T

In [15]:
final_response = model_with_tools.invoke(messages)
final_response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'pr7mnb07t', 'function': {'arguments': '{"location":"Boston"}', 'name': 'fetch_weather_data'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 286, 'total_tokens': 299, 'completion_time': 0.012156625, 'completion_tokens_details': None, 'prompt_time': 0.015823889, 'prompt_tokens_details': None, 'queue_time': 0.050274451, 'total_time': 0.027980514}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019be119-f19e-7931-9a7c-79656aaa02f2-0', tool_calls=[{'name': 'fetch_weather_data', 'args': {'location': 'Boston'}, 'id': 'pr7mnb07t', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 286, 'output_tokens': 13, 'total_tokens': 299})

## Why You Need an Agentic Loop

### The Problem You Encountered

Your final response had:
- ❌ Empty `content: ''`
- ❌ Still `tool_calls` (trying to call the tool again!)
- ❌ A message suggesting you use web scraping

This happened because **the model was called only once after adding the tool result**. The model needs to be called multiple times in a loop.

### The Flow Difference

**Without Loop (What you had - ❌):**
```
Iteration 1: Call model → Tool calls get_weather
Add tool result to messages
Iteration 2: Call model ONCE → Still calls tool (incomplete!)
```

**With Loop (What you need - ✅):**
```
Iteration 1: Call model → Tool calls get_weather
            Add tool result to messages
            Loop back ↻

Iteration 2: Call model → Sees tool result + can now respond
            No more tool calls
            Exit loop → Final response ready!
```

### Key Insight

The model's behavior:
- **Iteration 1**: "I need weather data, let me call the tool"
- **Iteration 2**: "I already have the weather data from iteration 1, I can now answer the user's question with actual content"

### The Loop Pattern

```python
while True:
    response = model.invoke(messages)
    
    # If no tool calls, we're done!
    if not response.tool_calls:
        return response  # ✅ Has actual content
    
    # Otherwise, execute tools and continue loop
    messages.append(response)
    for tool_call in response.tool_calls:
        result = execute_tool(tool_call)
        messages.append(result)
    # Loop continues - model called again with more context
```

This is the **agentic pattern** - an agent keeps acting (calling tools) until it reaches a goal (generating a response without tool calls).

In [ ]:

# Clear the 'tool' variable from previous cells to avoid conflicts
if 'tool' in dir():
    del tool

TypeError: 'dict' object is not callable

In [ ]:
# Bind (potentially multiple) tools to the model
model_with_tools = model.bind_tools([get_weather])

# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."